# 🔌 EV Charging Station - Data Science & Analytics Notebook

**Project**: Smart EV Charging Station Platform
**Purpose**: Exploratory Data Analysis, Feature Engineering, ML Model Training & Evaluation
**Created**: March 17, 2026

---

## Notebook Overview

This notebook demonstrates:
1. **Exploratory Data Analysis (EDA)** - Understanding charging data patterns
2. **Feature Engineering** - Creating ML-ready features
3. **ML Model Training** - Availability prediction & demand forecasting
4. **Model Evaluation** - Performance metrics and validation
5. **Predictions** - Real-world usage examples

The data flows through our production pipeline and can be used for:
- Predicting available charging slots
- Forecasting peak demand times
- Detecting anomalies (hardware failures, unusual patterns)
- Optimizing station management

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Set up visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ All libraries imported successfully')

## 2. Create Sample EV Charging Dataset

Generate synthetic historical charging data for demonstration

In [ ]:
# Create synthetic charging dataset for demonstration
np.random.seed(42)

# Generate 30 days of charging data
dates = pd.date_range(start='2026-02-16', periods=30*24, freq='H')

data = {
    'timestamp': dates,
    'station_id': np.random.choice(['STATION_001', 'STATION_002', 'STATION_003'], len(dates)),
    'latitude': np.random.choice([28.5244, 28.5354, 28.5454], len(dates)),
    'longitude': np.random.choice([77.2064, 77.2164, 77.2264], len(dates)),
    'temperature': 25 + np.random.normal(0, 5, len(dates)),
    'weather': np.random.choice(['sunny', 'cloudy', 'rainy'], len(dates)),
    'total_chargers': 10,
    'available_slots': np.random.randint(0, 11, len(dates)),
    'occupied_slots': np.random.randint(0, 11, len(dates)),
}

df = pd.DataFrame(data)

# Add demand patterns (higher during peak hours)
df['hour'] = df['timestamp'].dt.hour
peak_factor = np.where(df['hour'].isin([8, 9, 12, 13, 17, 18, 19]), 1.5, 0.8)
df['available_slots'] = (df['available_slots'] * peak_factor).astype(int)
df['available_slots'] = df['available_slots'].clip(0, 10)

print(f'✅ Dataset created: {len(df):,} rows')
print(f'📊 Shape: {df.shape}')
print(f'\n{df.head(10)}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Data Overview
print('📊 DATASET OVERVIEW')
print('='*50)
print(f'Total Records: {len(df):,}')
print(f'Date Range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Unique Stations: {df["station_id"].nunique()}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')

In [ ]:
# Statistical Summary
print('📈 STATISTICAL SUMMARY')
print('='*50)
print(df[['available_slots', 'occupied_slots', 'temperature']].describe())

In [ ]:
# Availability Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of available slots
axes[0].hist(df['available_slots'], bins=11, color='skyblue', edgecolor='black')
axes[0].set_title('Distribution of Available Charging Slots', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Available Slots')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

# Box plot by hour
df.boxplot(column='available_slots', by='hour', ax=axes[1])
axes[1].set_title('Availability by Hour of Day', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Available Slots')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('✅ Availability patterns visualized')

In [ ]:
# Time Series Analysis
hourly_avg = df.groupby('hour')['available_slots'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(hourly_avg['hour'], hourly_avg['mean'], marker='o', linewidth=2, label='Average Availability')
ax.fill_between(hourly_avg['hour'], 
                  hourly_avg['mean'] - hourly_avg['std'],
                  hourly_avg['mean'] + hourly_avg['std'],
                  alpha=0.3, label='±1 Std Dev')
ax.set_title('Average Charging Availability by Hour', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Available Slots')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Peak hours (lowest availability):')
peak_hours = hourly_avg.nsmallest(3, 'mean')[['hour', 'mean']]
for _, row in peak_hours.iterrows():
    print(f'  Hour {int(row["hour"]):02d}:00 - {row["mean"]:.1f} slots available')

In [ ]:
# Station Performance Comparison
station_stats = df.groupby('station_id').agg({
    'available_slots': ['mean', 'min', 'max', 'std'],
    'occupied_slots': 'mean'
}).round(2)

print('🏢 STATION PERFORMANCE METRICS')
print('='*70)
print(station_stats)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
station_avgs = df.groupby('station_id')['available_slots'].mean().sort_values()
station_avgs.plot(kind='bar', ax=ax, color='tomato', edgecolor='black')
ax.set_title('Average Availability per Station', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Available Slots')
ax.set_xlabel('Station ID')
plt.tight_layout()
plt.show()

## 4. Feature Engineering Pipeline Integration

Apply the production feature engineering pipeline

In [ ]:
# Create temporal features (matching production pipeline)
print('🔧 FEATURE ENGINEERING')
print('='*50)

df['day_of_week'] = df['timestamp'].dt.dayofweek  # 0=Monday, 6=Sunday
df['month'] = df['timestamp'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_business_hours'] = df['hour'].isin([9, 10, 11, 14, 15, 16]).astype(int)

# Create demand categories
df['occupancy_rate'] = df['occupied_slots'] / df['total_chargers']
df['availability_rate'] = df['available_slots'] / df['total_chargers']

# Demand level (0=Plenty, 1=Available, 2=Busy, 3=Full)
def categorize_demand(availability_rate):
    if availability_rate >= 0.6:
        return 0  # Plenty
    elif availability_rate >= 0.4:
        return 1  # Available
    elif availability_rate >= 0.2:
        return 2  # Busy
    else:
        return 3  # Full

df['demand_level'] = df['availability_rate'].apply(categorize_demand)

# Rolling statistics (1h and 6h windows)
df['rolling_mean_1h'] = df.groupby('station_id')['available_slots'].transform(
    lambda x: x.rolling(window=1, min_periods=1).mean()
)
df['rolling_mean_6h'] = df.groupby('station_id')['available_slots'].transform(
    lambda x: x.rolling(window=6, min_periods=1).mean()
)

print(f'✅ Features created: {len(df.columns)} features')
print(f'\nNew Features: {[
    "day_of_week", "month", "is_weekend", "is_business_hours",
    "occupancy_rate", "availability_rate", "demand_level",
    "rolling_mean_1h", "rolling_mean_6h"